In [3]:
import pandas as pd 
data = pd.read_csv("D:\\depi project\\t_data.csv")

In [ ]:
import pyodbc
from sqlalchemy import create_engine
server ='DESKTOP-2HBAHIM\SQLEXPRESS'
database = 'SuperMarkets'
engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
)

print("Connected successfully!")

In [ ]:
product = data[['aisle','product_name']].drop_duplicates().reset_index(drop=True)

print(product)

In [ ]:
product.to_sql(
    'Products',
    engine,
    index=False,
    if_exists="append"
)


In [ ]:
daily_sales = data.groupby(['product_id','store_id','transaction_date']).agg({
    'quantity': 'sum',
    'final_amount': 'sum'
}).reset_index()
daily_sales

daily_sales.rename(columns={
    "transaction_date": "sale_date",
    "quantity" : "total_quantity",
      "final_amount":"total_revenue"
}, inplace=True)
daily_sales

In [ ]:
daily_sales.to_sql('DailySales',engine,index=False,if_exists='append')

In [ ]:
store = data[['store_name']].drop_duplicates()
store

In [ ]:
store.to_sql(
    'Stores',
    engine,
    index=False,
    if_exists='append'
)

In [ ]:
product_db = pd.read_sql("select * from products",engine)
store_db = pd.read_sql("select * from stores" , engine)
data = data.merge(product_db , on="product_name")
data = data.merge(store_db, on='store_name')


In [ ]:
sales = data[['store_id',
'product_id',
'transaction_date',
'quantity',
'discount_amount',
'loyalty_points',
'unit_price']]

In [ ]:
sales.rename(columns={
    "transaction_date": "sale_date"
}, inplace=True)

In [ ]:
sales.to_sql('sales' , 
             engine,
             index=False,
             if_exists="append")